# ✈️ AirFly Insights — Airline Operations Analytics

**Project objective:** Explore airline operational data to understand flight volume, airline performance, airport and route activity, delays, diversions, and cancellations.

**Workflow:** Data loading → data quality checks → cleaning → feature engineering → KPIs → exploratory visualizations → interactive notebook dashboard → business insights → Excel export.

> **Dataset note:** This notebook uses the 10,000-row `flight_data_2024_sample.csv` sample. Findings describe this sample and should not automatically be generalized to the entire airline industry or the full source dataset.

**How to use:** Run the notebook from top to bottom. The dataset-loading cell first checks Google Drive; if the CSV is not there, it offers an upload fallback.

Import libraries and upload the dataset

In [ ]:
# STEP 1: IMPORT LIBRARIES AND LOAD THE DATASET

import os
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files, drive

# Preferred: keep the dataset in Google Drive so it is not uploaded
# again every time a Colab runtime restarts.
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/AirFly-Insights')
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_NAME = 'flight_data_2024_sample.csv'
csv_path = DATA_DIR / CSV_NAME

# If the CSV is not in Drive yet, upload either the CSV or its ZIP file once.
if not csv_path.exists():
    print(f"Place {CSV_NAME} in: {DATA_DIR}")
    print("If it is not there yet, choose the CSV or ZIP file in the upload window.")
    uploaded = files.upload()

    for uploaded_name in uploaded:
        if uploaded_name.lower().endswith('.csv'):
            # Save the uploaded CSV into Drive so it persists after runtime resets.
            import shutil
            shutil.copy2(uploaded_name, csv_path)

        elif uploaded_name.lower().endswith('.zip'):
            with zipfile.ZipFile(uploaded_name, 'r') as archive:
                matches = [
                    name for name in archive.namelist()
                    if Path(name).name == CSV_NAME
                ]
                if matches:
                    with archive.open(matches[0]) as source, open(csv_path, 'wb') as target:
                        target.write(source.read())

    if not csv_path.exists():
        raise FileNotFoundError(
            f"{CSV_NAME} was not found. Upload the sample CSV or a ZIP containing it."
        )

df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Source:", csv_path)
print("Shape:", df.shape)
print("Columns:", len(df.columns))

Explore the dataset

In [ ]:

# STEP 2: Explore the dataset

# 1. Display first 5 records
print("FIRST 5 ROWS")
display(df.head())

# 2. Display last 5 records
print("LAST 5 ROWS")
display(df.tail())

# 3. Dataset dimensions
print("DATASET SHAPE:", df.shape)

# 4. Column names
print("\nCOLUMN NAMES:")
print(df.columns.tolist())

# 5. Dataset information
print("\nDATASET INFORMATION:")
df.info()

# 6. Statistical summary
print("\nSTATISTICAL SUMMARY:")
display(df.describe())

# 7. Missing values
print("\nMISSING VALUES:")
missing = df.isnull().sum()
display(missing[missing > 0].sort_values(ascending=False))


DATA CLEANING

In [ ]:
# STEP 3: DATA CLEANING

# Keep the original dataset safe.
df_clean = df.copy()

# Convert date and numeric columns to suitable types.
df_clean['fl_date'] = pd.to_datetime(df_clean['fl_date'], errors='coerce')

numeric_columns = [
    'cancelled', 'diverted', 'crs_dep_time', 'dep_time',
    'dep_delay', 'crs_arr_time', 'arr_time', 'arr_delay',
    'air_time', 'actual_elapsed_time', 'distance',
    'carrier_delay', 'weather_delay', 'nas_delay',
    'security_delay', 'late_aircraft_delay'
]
for column in numeric_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_numeric(df_clean[column], errors='coerce')

# Cancellation/diversion flags are operational indicators.
# Missing flags are treated as 0; missing delay values are NOT replaced with 0.
for flag in ['cancelled', 'diverted']:
    df_clean[flag] = df_clean[flag].fillna(0).astype('int8')

# Blank cancellation codes for non-cancelled flights are labelled clearly.
df_clean.loc[df_clean['cancelled'] == 0, 'cancellation_code'] = 'Not Cancelled'

# Remove exact duplicate records.
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
duplicates_removed = rows_before - len(df_clean)

# Assign one mutually exclusive operational status to each record.
conditions = [
    df_clean['cancelled'] == 1,
    df_clean['diverted'] == 1
]
choices = ['Cancelled', 'Diverted']
df_clean['flight_status'] = np.select(
    conditions, choices, default='Completed'
)

print("CLEANING COMPLETED")
print("Exact duplicates removed:", duplicates_removed)
print("Cleaned shape:", df_clean.shape)
print("\nFlight status:")
display(df_clean['flight_status'].value_counts().rename_axis('Status').to_frame('Flights'))

print("\nMissing values (top 15):")
display(df_clean.isnull().sum().sort_values(ascending=False).head(15).to_frame('Missing Values'))

print("\nCleaned preview:")
display(df_clean.head())

DATE FEATURES

In [ ]:

# STEP 4.1: DATE FEATURES

# Extract year
df_clean['year'] = df_clean['fl_date'].dt.year

# Extract month number
df_clean['month'] = df_clean['fl_date'].dt.month

# Extract month name
df_clean['month_name'] = df_clean['fl_date'].dt.month_name()

# Extract weekday name
df_clean['weekday_name'] = df_clean['fl_date'].dt.day_name()

# Extract day of month
df_clean['day'] = df_clean['fl_date'].dt.day

print("DATE FEATURES CREATED!")

display(
    df_clean[
        ['fl_date', 'year', 'month',
         'month_name', 'weekday_name', 'day']
    ].head(10)
)


In [ ]:

# STEP 4.2: DEPARTURE HOUR

# Convert scheduled departure time into numeric format
dep_time_numeric = pd.to_numeric(
    df_clean['crs_dep_time'], errors='coerce'
)

# Extract hour
df_clean['departure_hour'] = (
    dep_time_numeric // 100
).where(dep_time_numeric.notna())

# Handle invalid time values
df_clean.loc[
    ~df_clean['departure_hour'].between(0, 23),
    'departure_hour'
] = np.nan

print("DEPARTURE HOUR CREATED!")

display(
    df_clean[
        ['crs_dep_time', 'departure_hour']
    ].head(10)
)


In [ ]:

# STEP 4.3: ROUTE FEATURE

df_clean['route'] = (
    df_clean['origin'].astype(str)
    + ' → '
    + df_clean['dest'].astype(str)
)

print("ROUTE FEATURE CREATED!")

display(
    df_clean[['origin', 'dest', 'route']].head(10)
)


In [ ]:

# STEP 4.4: DELAY CATEGORY

conditions = [
    df_clean['cancelled'] == 1,
    df_clean['diverted'] == 1,
    df_clean['arr_delay'] <= 15,
    df_clean['arr_delay'].between(16, 60),
    df_clean['arr_delay'].between(61, 180),
    df_clean['arr_delay'] > 180
]

choices = [
    'Cancelled',
    'Diverted',
    'On Time',
    'Minor Delay',
    'Major Delay',
    'Severe Delay'
]

df_clean['delay_category'] = np.select(
    conditions,
    choices,
    default='Unknown'
)

print("DELAY CATEGORIES CREATED!")

display(df_clean['delay_category'].value_counts())


In [ ]:

# STEP 4.5: DISTANCE CATEGORY

df_clean['distance_category'] = pd.cut(
    df_clean['distance'],
    bins=[0, 500, 1500, float('inf')],
    labels=['Short Haul', 'Medium Haul', 'Long Haul']
)

display(
    df_clean[
        ['distance', 'distance_category']
    ].head(10)
)


In [ ]:

# STEP 4.6: VERIFY NEW FEATURES

new_features = [
    'month_name',
    'weekday_name',
    'departure_hour',
    'route',
    'delay_category',
    'distance_category'
]

print("NEW FEATURES CREATED:")
display(df_clean[new_features].head(10))

print("\nDATASET SHAPE:")
print(df_clean.shape)

print("\nDATA TYPES:")
print(df_clean[new_features].dtypes)


In [ ]:

# STEP 5.1: OVERALL KPI CALCULATIONS

total_flights = len(df_clean)

completed = df_clean[
    df_clean['flight_status'] == 'Completed'
]

cancelled = df_clean[
    df_clean['flight_status'] == 'Cancelled'
]

diverted = df_clean[
    df_clean['flight_status'] == 'Diverted'
]

# Only flights with a recorded arrival delay
valid_arrivals = completed[
    completed['arr_delay'].notna()
]

# On-time arrivals: delay of 15 minutes or less
on_time = valid_arrivals[
    valid_arrivals['arr_delay'] <= 15
]

# Flights delayed by more than 15 minutes
delayed = valid_arrivals[
    valid_arrivals['arr_delay'] > 15
]

# Calculate KPIs
on_time_percentage = (
    len(on_time) / len(valid_arrivals) * 100
    if len(valid_arrivals) else 0
)

cancellation_percentage = (
    len(cancelled) / total_flights * 100
    if total_flights else 0
)

diversion_percentage = (
    len(diverted) / total_flights * 100
    if total_flights else 0
)

average_arrival_delay = valid_arrivals[
    'arr_delay'
].mean()

print("✈️ AIRFLY INSIGHTS — OVERALL KPIs")
print("-" * 45)

print(f"Total Flights: {total_flights:,}")
print(f"Completed Flights: {len(completed):,}")
print(f"On-Time Arrivals: {on_time_percentage:.2f}%")
print(f"Average Arrival Delay: {average_arrival_delay:.2f} minutes")
print(f"Delayed Flights: {len(delayed):,}")
print(f"Cancelled Flights: {len(cancelled):,}")
print(f"Cancellation Rate: {cancellation_percentage:.2f}%")
print(f"Diverted Flights: {len(diverted):,}")
print(f"Diversion Rate: {diversion_percentage:.2f}%")


In [ ]:

# STEP 5.2: AIRLINE PERFORMANCE ANALYSIS

airline_performance = (
    df_clean.groupby('op_unique_carrier')
    .agg(
        Total_Flights=('op_unique_carrier', 'size'),
        Average_Departure_Delay=('dep_delay', 'mean'),
        Average_Arrival_Delay=('arr_delay', 'mean'),
        Cancelled_Flights=('cancelled', 'sum'),
        Diverted_Flights=('diverted', 'sum')
    )
    .reset_index()
)

# Calculate cancellation percentage
airline_performance['Cancellation_Rate'] = (
    airline_performance['Cancelled_Flights']
    / airline_performance['Total_Flights'] * 100
)

# Sort by average arrival delay
airline_performance = airline_performance.sort_values(
    'Average_Arrival_Delay',
    ascending=False
)

display(airline_performance.round(2))


In [ ]:

# STEP 5.3: DELAY CAUSE ANALYSIS

delay_causes = {
    'Carrier': 'carrier_delay',
    'Weather': 'weather_delay',
    'NAS': 'nas_delay',
    'Security': 'security_delay',
    'Late Aircraft': 'late_aircraft_delay'
}

delay_analysis = pd.DataFrame({
    'Delay Cause': delay_causes.keys(),
    'Total Delay Minutes': [
        pd.to_numeric(df_clean[col], errors='coerce').sum()
        for col in delay_causes.values()
    ]
})

delay_analysis = delay_analysis.sort_values(
    'Total Delay Minutes',
    ascending=False
)

display(delay_analysis)

top_delay_cause = delay_analysis.iloc[0]['Delay Cause']

print("Most significant recorded delay cause:", top_delay_cause)


In [ ]:

# STEP 5.4: DELAY CAUSE VISUALIZATION

import plotly.express as px

fig = px.bar(
    delay_analysis,
    x='Delay Cause',
    y='Total Delay Minutes',
    color='Total Delay Minutes',
    title='✈️ Major Causes of Flight Delays',
    text_auto='.0f',
    template='plotly_dark',
    color_continuous_scale='Teal'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Delay Cause',
    yaxis_title='Total Delay Minutes',
    height=500
)

fig.show()


In [ ]:

# STEP 5.5: KPI SUMMARY TABLE

kpi_summary = pd.DataFrame({
    'Metric': [
        'Total Flights',
        'Completed Flights',
        'On-Time Arrival (%)',
        'Average Arrival Delay (min)',
        'Delayed Flights',
        'Cancelled Flights',
        'Cancellation Rate (%)',
        'Diverted Flights',
        'Diversion Rate (%)'
    ],
    'Value': [
        total_flights,
        len(completed),
        round(on_time_percentage, 2),
        round(average_arrival_delay, 2),
        len(delayed),
        len(cancelled),
        round(cancellation_percentage, 2),
        len(diverted),
        round(diversion_percentage, 2)
    ]
})

display(kpi_summary)


In [ ]:

# STEP 6: IMPORT VISUALIZATION LIBRARIES

import plotly.express as px
import plotly.graph_objects as go

# Professional chart theme
CHART_THEME = "plotly_dark"

print("Visualization libraries imported successfully!")


In [ ]:

# MONTHLY FLIGHT TRAFFIC

monthly_flights = (
    df_clean.groupby(['month', 'month_name'])
    .size()
    .reset_index(name='Total Flights')
    .sort_values('month')
)

fig = px.line(
    monthly_flights,
    x='month_name',
    y='Total Flights',
    markers=True,
    text='Total Flights',
    title='Monthly Flight Traffic — 2024',
    template=CHART_THEME
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=9),
    textposition='top center'
)

fig.update_layout(
    title_x=0.5,
    height=500,
    xaxis_title='Month',
    yaxis_title='Number of Flights'
)

fig.show()


In [ ]:

# AIRLINE FLIGHT VOLUME

airline_flights = (
    df_clean.groupby('op_unique_carrier')
    .size()
    .reset_index(name='Total Flights')
    .sort_values('Total Flights', ascending=False)
)

fig = px.bar(
    airline_flights,
    x='op_unique_carrier',
    y='Total Flights',
    color='Total Flights',
    text_auto=True,
    title='Flight Volume by Airline',
    template=CHART_THEME,
    color_continuous_scale='Teal'
)

fig.update_layout(
    title_x=0.5,
    height=550,
    xaxis_title='Airline Code',
    yaxis_title='Total Flights'
)

fig.show()


In [ ]:

# DEPARTURE VS ARRIVAL DELAY

delay_comparison = (
    df_clean.groupby('op_unique_carrier')
    .agg(
        Departure_Delay=('dep_delay', 'mean'),
        Arrival_Delay=('arr_delay', 'mean')
    )
    .reset_index()
)

delay_comparison = delay_comparison.sort_values(
    'Arrival_Delay',
    ascending=False
)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=delay_comparison['op_unique_carrier'],
    y=delay_comparison['Departure_Delay'],
    name='Departure Delay'
))

fig.add_trace(go.Bar(
    x=delay_comparison['op_unique_carrier'],
    y=delay_comparison['Arrival_Delay'],
    name='Arrival Delay'
))

fig.update_layout(
    title='Average Departure vs Arrival Delay by Airline',
    title_x=0.5,
    barmode='group',
    template=CHART_THEME,
    height=550,
    xaxis_title='Airline',
    yaxis_title='Average Delay (Minutes)'
)

fig.show()


In [ ]:

# AIRPORT TRAFFIC ANALYSIS

origin_traffic = (
    df_clean['origin']
    .value_counts()
    .rename_axis('Airport')
    .reset_index(name='Flights')
)

destination_traffic = (
    df_clean['dest']
    .value_counts()
    .rename_axis('Airport')
    .reset_index(name='Flights')
)

airport_traffic = pd.concat(
    [origin_traffic, destination_traffic]
)

airport_traffic = (
    airport_traffic.groupby('Airport')['Flights']
    .sum()
    .reset_index()
    .sort_values('Flights', ascending=False)
    .head(15)
)

fig = px.bar(
    airport_traffic.sort_values('Flights'),
    x='Flights',
    y='Airport',
    orientation='h',
    color='Flights',
    title='Top 15 Busiest Airports',
    template=CHART_THEME,
    color_continuous_scale='Viridis'
)

fig.update_layout(
    title_x=0.5,
    height=650,
    xaxis_title='Total Airport Movements',
    yaxis_title='Airport'
)

fig.show()


In [ ]:

# TOP 15 FLIGHT ROUTES

route_analysis = (
    df_clean.groupby('route')
    .agg(
        Total_Flights=('route', 'size'),
        Average_Arrival_Delay=('arr_delay', 'mean')
    )
    .reset_index()
    .sort_values('Total_Flights', ascending=False)
    .head(15)
)

fig = px.bar(
    route_analysis.sort_values('Total_Flights'),
    x='Total_Flights',
    y='route',
    orientation='h',
    color='Average_Arrival_Delay',
    title='Top 15 Most Popular Flight Routes',
    template=CHART_THEME,
    color_continuous_scale='Turbo',
    hover_data=['Average_Arrival_Delay']
)

fig.update_layout(
    title_x=0.5,
    height=650,
    xaxis_title='Number of Flights',
    yaxis_title='Flight Route'
)

fig.show()


In [ ]:

# FLIGHT STATUS DISTRIBUTION

status_data = (
    df_clean['flight_status']
    .value_counts()
    .reset_index()
)

status_data.columns = ['Flight Status', 'Count']

fig = px.pie(
    status_data,
    names='Flight Status',
    values='Count',
    hole=0.55,
    title='Flight Status Distribution',
    template=CHART_THEME
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label'
)

fig.update_layout(
    title_x=0.5,
    height=500
)

fig.show()


In [ ]:

# MONTHLY CANCELLATION ANALYSIS

cancellation_monthly = (
    df_clean.groupby('month_name')
    .agg(
        Total_Flights=('cancelled', 'size'),
        Cancelled_Flights=('cancelled', 'sum')
    )
    .reset_index()
)

# Sort months chronologically
month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

cancellation_monthly['month_name'] = pd.Categorical(
    cancellation_monthly['month_name'],
    categories=month_order,
    ordered=True
)

cancellation_monthly = cancellation_monthly.sort_values(
    'month_name'
)

cancellation_monthly['Cancellation_Rate'] = (
    cancellation_monthly['Cancelled_Flights']
    / cancellation_monthly['Total_Flights'] * 100
)

fig = px.bar(
    cancellation_monthly,
    x='month_name',
    y='Cancellation_Rate',
    color='Cancellation_Rate',
    text_auto='.2f',
    title='Monthly Flight Cancellation Rate',
    template=CHART_THEME,
    color_continuous_scale='Reds'
)

fig.update_layout(
    title_x=0.5,
    height=500,
    xaxis_title='Month',
    yaxis_title='Cancellation Rate (%)'
)

fig.show()


In [ ]:

# STEP 7.1: DASHBOARD LIBRARIES

!pip -q install ipywidgets plotly

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("Dashboard libraries are ready!")


In [ ]:

# STEP 7.2: INTERACTIVE FILTERS

airline_options = ['All Airlines'] + sorted(
    df_clean['op_unique_carrier'].dropna().unique().tolist()
)

month_options = ['All Months'] + [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

status_options = ['All Statuses', 'Completed', 'Cancelled', 'Diverted']

airline_filter = widgets.Dropdown(
    options=airline_options,
    value='All Airlines',
    description='Airline:',
    layout=widgets.Layout(width='250px')
)

month_filter = widgets.Dropdown(
    options=month_options,
    value='All Months',
    description='Month:',
    layout=widgets.Layout(width='250px')
)

status_filter = widgets.Dropdown(
    options=status_options,
    value='All Statuses',
    description='Status:',
    layout=widgets.Layout(width='250px')
)

theme_filter = widgets.ToggleButtons(
    options=['Dark', 'Light'],
    value='Dark',
    description='Theme:',
    button_style='info'
)

print("Interactive filters created!")


In [ ]:

# STEP 7.3: DASHBOARD ENGINE

dashboard_output = widgets.Output()

def create_dashboard(*args):

    with dashboard_output:
        clear_output(wait=True)

        # Get selected filter values
        selected_airline = airline_filter.value
        selected_month = month_filter.value
        selected_status = status_filter.value
        selected_theme = theme_filter.value

        theme = (
            'plotly_dark'
            if selected_theme == 'Dark'
            else 'plotly_white'
        )

        # Apply filters
        filtered = df_clean.copy()

        if selected_airline != 'All Airlines':
            filtered = filtered[
                filtered['op_unique_carrier'] == selected_airline
            ]

        if selected_month != 'All Months':
            filtered = filtered[
                filtered['month_name'] == selected_month
            ]

        if selected_status != 'All Statuses':
            filtered = filtered[
                filtered['flight_status'] == selected_status
            ]

        if filtered.empty:
            display(HTML(
                "<h3 style='color:orange'>"
                "No flights match the selected filters."
                "</h3>"
            ))
            return

        # KPI calculations
        total = len(filtered)

        completed_flights = filtered[
            filtered['flight_status'] == 'Completed'
        ]

        valid = completed_flights[
            completed_flights['arr_delay'].notna()
        ]

        on_time = valid[valid['arr_delay'] <= 15]

        on_time_rate = (
            len(on_time) / len(valid) * 100
            if len(valid) else 0
        )

        avg_delay = (
            valid['arr_delay'].mean()
            if len(valid) else 0
        )

        cancel_rate = (
            filtered['cancelled'].sum() / total * 100
        )

        diverted_count = filtered['diverted'].sum()

        # Dashboard heading
        display(HTML(f"""
        <div style="padding:20px;
                    border-radius:12px;
                    background:{'#172554' if selected_theme == 'Dark' else '#eaf2ff'};
                    color:{'white' if selected_theme == 'Dark' else '#172554'};
                    text-align:center;">
            <h1>✈️ AirFly Insights</h1>
            <p>Airline Operations Analytics Dashboard</p>
            <p>Interactive analysis of 2024 flight operations</p>
        </div>
        """))

        # KPI cards
        kpis = [
            ('Total Flights', f'{total:,}', '✈️'),
            ('On-Time Arrival', f'{on_time_rate:.1f}%', '⏱️'),
            ('Average Arrival Delay', f'{avg_delay:.1f} min', '📊'),
            ('Cancellation Rate', f'{cancel_rate:.2f}%', '❌'),
            ('Diverted Flights', f'{diverted_count:,}', '↪️')
        ]

        cards_html = ""

        for label, value, emoji in kpis:
            cards_html += f"""
            <div style="
                flex:1;
                min-width:140px;
                padding:18px;
                border-radius:12px;
                background:{'#243247' if selected_theme == 'Dark' else '#f1f5f9'};
                text-align:center;
                margin:5px;">
                <div style="font-size:25px">{emoji}</div>
                <div style="font-size:23px;font-weight:bold">{value}</div>
                <div style="font-size:13px">{label}</div>
            </div>
            """

        display(HTML(f"""
        <div style="display:flex;flex-wrap:wrap;margin-top:15px;">
            {cards_html}
        </div>
        """))

        # Common chart layout
        chart_height = 420

        # 1. Monthly trend
        monthly = (
            filtered.groupby('month')
            .size()
            .reset_index(name='Flights')
        )

        fig1 = px.line(
            monthly,
            x='month',
            y='Flights',
            markers=True,
            title='Monthly Flight Traffic',
            template=theme
        )

        fig1.update_layout(height=chart_height)
        fig1.update_xaxes(
            tickmode='array',
            tickvals=list(range(1, 13)),
            ticktext=[
                'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
            ]
        )

        # 2. Airline performance
        airline_data = (
            filtered.groupby('op_unique_carrier')
            .size()
            .reset_index(name='Flights')
            .sort_values('Flights', ascending=False)
        )

        fig2 = px.bar(
            airline_data,
            x='op_unique_carrier',
            y='Flights',
            color='Flights',
            title='Flight Volume by Airline',
            template=theme
        )

        fig2.update_layout(height=chart_height)

        # 3. Flight status
        status_data = (
            filtered['flight_status']
            .value_counts()
            .reset_index()
        )

        status_data.columns = ['Status', 'Flights']

        fig3 = px.pie(
            status_data,
            names='Status',
            values='Flights',
            hole=0.55,
            title='Flight Status Distribution',
            template=theme
        )

        fig3.update_layout(height=chart_height)

        # 4. Delay causes
        cause_columns = {
            'Carrier': 'carrier_delay',
            'Weather': 'weather_delay',
            'NAS': 'nas_delay',
            'Security': 'security_delay',
            'Late Aircraft': 'late_aircraft_delay'
        }

        cause_data = pd.DataFrame({
            'Cause': list(cause_columns.keys()),
            'Delay Minutes': [
                pd.to_numeric(filtered[col], errors='coerce').sum()
                for col in cause_columns.values()
            ]
        })

        fig4 = px.bar(
            cause_data,
            x='Cause',
            y='Delay Minutes',
            color='Delay Minutes',
            title='Delay Causes',
            template=theme
        )

        fig4.update_layout(height=chart_height)

        # 5. Top routes
        routes = (
            filtered.groupby('route')
            .size()
            .reset_index(name='Flights')
            .sort_values('Flights', ascending=False)
            .head(10)
            .sort_values('Flights')
        )

        fig5 = px.bar(
            routes,
            x='Flights',
            y='route',
            orientation='h',
            title='Top 10 Flight Routes',
            template=theme
        )

        fig5.update_layout(height=chart_height)

        # 6. Airport traffic
        airports = pd.concat([
            filtered['origin'].value_counts(),
            filtered['dest'].value_counts()
        ], axis=1).fillna(0)

        airports.columns = ['Origin', 'Destination']

        airports['Total Movements'] = (
            airports['Origin'] + airports['Destination']
        )

        airports = (
            airports.sort_values(
                'Total Movements', ascending=False
            ).head(10).reset_index(names='Airport')
        )

        fig6 = px.bar(
            airports.sort_values('Total Movements'),
            x='Total Movements',
            y='Airport',
            orientation='h',
            title='Top 10 Busiest Airports',
            template=theme
        )

        fig6.update_layout(height=chart_height)

        # Display charts
        display(HTML("<h2>📈 Flight Trends & Performance</h2>"))
        display(fig1, fig2)

        display(HTML("<h2>🛫 Flight Status & Delay Analysis</h2>"))
        display(fig3, fig4)

        display(HTML("<h2>🌎 Routes & Airports</h2>"))
        display(fig5, fig6)

        # Final summary
        display(HTML(f"""
        <div style="padding:15px;
                    border-radius:10px;
                    background:{'#243247' if selected_theme == 'Dark' else '#f1f5f9'};
                    margin-top:15px;">
            <h3>Dashboard Summary</h3>
            <p>Selected records: <b>{total:,}</b></p>
            <p>Completed flights: <b>{len(completed_flights):,}</b></p>
            <p>Delayed arrivals (>15 min): <b>{len(valid) - len(on_time):,}</b></p>
            <p>Cancelled flights: <b>{int(filtered['cancelled'].sum()):,}</b></p>
        </div>
        """))


In [ ]:

# STEP 7.4: CONNECT FILTERS

for control in [
    airline_filter,
    month_filter,
    status_filter,
    theme_filter
]:
    control.observe(create_dashboard, names='value')

# Arrange filters horizontally
filter_panel = widgets.VBox([
    widgets.HTML("<h3>🎛️ Dashboard Controls</h3>"),
    widgets.HBox([
        airline_filter,
        month_filter
    ]),
    widgets.HBox([
        status_filter,
        theme_filter
    ])
])

# Display dashboard
display(filter_panel)
display(dashboard_output)

# Generate initial dashboard
create_dashboard()


In [ ]:

# STEP 8.1: AUTOMATIC BUSINESS INSIGHTS

insights = []

# 1. Busiest airline
airline_counts = df_clean['op_unique_carrier'].value_counts()

if not airline_counts.empty:
    busiest_airline = airline_counts.idxmax()
    busiest_count = airline_counts.max()

    insights.append(
        f"✈️ {busiest_airline} operates the highest number "
        f"of flights in our sample ({busiest_count:,})."
    )

# 2. Busiest month
month_counts = df_clean['month_name'].value_counts()

if not month_counts.empty:
    busiest_month = month_counts.idxmax()

    insights.append(
        f"📅 {busiest_month} has the highest flight volume "
        f"in our sample."
    )

# 3. Busiest route
route_counts = df_clean['route'].value_counts()

if not route_counts.empty:
    busiest_route = route_counts.idxmax()

    insights.append(
        f"🌎 The most frequently recorded route is "
        f"{busiest_route} ({route_counts.max():,} flights)."
    )

# 4. Highest delay cause
if not delay_analysis.empty:
    top_cause = delay_analysis.iloc[0]['Delay Cause']
    top_cause_minutes = delay_analysis.iloc[0]['Total Delay Minutes']

    insights.append(
        f"⏱️ {top_cause} accounts for the highest recorded "
        f"delay minutes ({top_cause_minutes:,.0f})."
    )

# 5. Cancellation rate
total = len(df_clean)
cancel_rate = df_clean['cancelled'].sum() / total * 100

insights.append(
    f"❌ The cancellation rate in this sample is "
    f"{cancel_rate:.2f}%."
)

# 6. Average arrival delay
valid_delays = df_clean.loc[
    (df_clean['flight_status'] == 'Completed'),
    'arr_delay'
].dropna()

if not valid_delays.empty:
    avg_delay = valid_delays.mean()

    insights.append(
        f"📊 Average arrival delay among completed flights "
        f"with recorded data is {avg_delay:.2f} minutes."
    )

# Display insights
print("✈️ AIRFLY INSIGHTS — AUTOMATED BUSINESS FINDINGS")
print("=" * 65)

for i, insight in enumerate(insights, start=1):
    print(f"\n{i}. {insight}")


In [ ]:

# STEP 8.2: WEEKDAY DELAY ANALYSIS

weekday_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

weekday_analysis = (
    df_clean[df_clean['flight_status'] == 'Completed']
    .groupby('weekday_name')
    .agg(
        Total_Flights=('weekday_name', 'size'),
        Average_Departure_Delay=('dep_delay', 'mean'),
        Average_Arrival_Delay=('arr_delay', 'mean')
    )
    .reindex(weekday_order)
    .reset_index()
)

display(weekday_analysis.round(2))

fig = px.bar(
    weekday_analysis,
    x='weekday_name',
    y='Average_Arrival_Delay',
    color='Average_Arrival_Delay',
    title='Average Arrival Delay by Weekday',
    template='plotly_dark',
    color_continuous_scale='Turbo',
    hover_data=['Total_Flights']
)

fig.update_layout(
    title_x=0.5,
    height=500,
    xaxis_title='Weekday',
    yaxis_title='Average Arrival Delay (Minutes)'
)

fig.show()


In [ ]:

# STEP 8.3: HOURLY DEPARTURE ANALYSIS

hourly_analysis = (
    df_clean[df_clean['flight_status'] == 'Completed']
    .groupby('departure_hour')
    .agg(
        Total_Flights=('departure_hour', 'size'),
        Average_Departure_Delay=('dep_delay', 'mean'),
        Average_Arrival_Delay=('arr_delay', 'mean')
    )
    .reset_index()
)

hourly_analysis = hourly_analysis.dropna(
    subset=['departure_hour']
)

fig = px.line(
    hourly_analysis,
    x='departure_hour',
    y='Average_Departure_Delay',
    markers=True,
    title='Average Departure Delay by Hour',
    template='plotly_dark',
    hover_data=['Total_Flights']
)

fig.update_layout(
    title_x=0.5,
    height=500,
    xaxis_title='Scheduled Departure Hour',
    yaxis_title='Average Departure Delay (Minutes)',
    xaxis=dict(dtick=1)
)

fig.show()

display(hourly_analysis.round(2))


In [ ]:
# STEP 8.4: ARRIVAL DELAY SEVERITY ANALYSIS

# This chart includes Unknown so completed flights without a recorded
# arrival delay are not silently omitted from the distribution.
severity_order = [
    'On Time', 'Minor Delay', 'Major Delay',
    'Severe Delay', 'Unknown'
]

completed_categories = df_clean.loc[
    df_clean['flight_status'] == 'Completed', 'delay_category'
]

severity_data = (
    completed_categories.value_counts()
    .reindex(severity_order, fill_value=0)
    .rename_axis('Delay Category')
    .reset_index(name='Flights')
)

display(severity_data)

fig = px.pie(
    severity_data,
    names='Delay Category',
    values='Flights',
    hole=0.55,
    title='Arrival Delay Severity — Completed Flights',
    template='plotly_dark'
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(title_x=0.5, height=500)
fig.show()

In [ ]:

# STEP 8.5: CANCELLATION REASON ANALYSIS

cancelled_data = df_clean[
    df_clean['cancelled'] == 1
].copy()

if not cancelled_data.empty:

    cancellation_reasons = (
        cancelled_data['cancellation_code']
        .fillna('Unknown')
        .value_counts()
        .reset_index()
    )

    cancellation_reasons.columns = [
        'Cancellation Code',
        'Flights'
    ]

    display(cancellation_reasons)

    fig = px.bar(
        cancellation_reasons,
        x='Cancellation Code',
        y='Flights',
        color='Flights',
        text_auto=True,
        title='Flight Cancellations by Reason',
        template='plotly_dark',
        color_continuous_scale='Reds'
    )

    fig.update_layout(
        title_x=0.5,
        height=450
    )

    fig.show()

else:
    print("No cancelled flights found in the selected dataset.")


In [ ]:

# STEP 8.6: EXPORT EXCEL REPORT

# Prepare KPI data
kpi_report = pd.DataFrame({
    'Metric': [
        'Total Flights',
        'Completed Flights',
        'Cancelled Flights',
        'Diverted Flights',
        'On-Time Arrival Percentage',
        'Average Arrival Delay',
        'Cancellation Percentage'
    ],
    'Value': [
        len(df_clean),
        (df_clean['flight_status'] == 'Completed').sum(),
        df_clean['cancelled'].sum(),
        df_clean['diverted'].sum(),
        round(on_time_percentage, 2),
        round(average_arrival_delay, 2),
        round(cancellation_percentage, 2)
    ]
})

# Export all analysis tables
excel_file = 'AirFly_Insights_Analysis_Report.xlsx'

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:

    kpi_report.to_excel(
        writer, sheet_name='KPIs', index=False
    )

    airline_performance.to_excel(
        writer, sheet_name='Airline Performance', index=False
    )

    delay_analysis.to_excel(
        writer, sheet_name='Delay Causes', index=False
    )

    weekday_analysis.to_excel(
        writer, sheet_name='Weekday Analysis', index=False
    )

    hourly_analysis.to_excel(
        writer, sheet_name='Hourly Analysis', index=False
    )

    severity_data.to_excel(
        writer, sheet_name='Delay Severity', index=False
    )

    df_clean.to_excel(
        writer, sheet_name='Cleaned Data', index=False
    )

print("Excel report generated successfully!")
print("File:", excel_file)


In [ ]:

# DOWNLOAD EXCEL REPORT

from google.colab import files

files.download('AirFly_Insights_Analysis_Report.xlsx')


In [ ]:

# STEP 8.7: MAP CANCELLATION CODES

cancellation_map = {
    'A': 'Carrier',
    'B': 'Weather',
    'C': 'National Airspace System',
    'D': 'Security',
    'Not Cancelled': 'Not Cancelled'
}

cancelled_data['Cancellation Reason'] = (
    cancelled_data['cancellation_code']
    .map(cancellation_map)
    .fillna('Unknown')
)

cancellation_summary = (
    cancelled_data['Cancellation Reason']
    .value_counts()
    .reset_index()
)

cancellation_summary.columns = [
    'Cancellation Reason', 'Flights'
]

display(cancellation_summary)


## 9. Streamlit deployment (separate from notebook analysis)

The cells above contain the analytical workflow and the interactive `ipywidgets` dashboard that runs inside Colab.

The public Streamlit website is a separate application built from `AirFly_app.py`. For a persistent public link, deploy that app from the GitHub repository using Streamlit Community Cloud. Do not rely on a Colab runtime or Cloudflare quick tunnel for a resume link.

**Repository files expected:**
- `AirFly_app.py` — Streamlit application
- `flight_data_2024_sample.csv` — dataset used by the app
- `requirements.txt` — Python dependencies
- `AirFly_Insights.ipynb` — this analysis notebook

### Analytical interpretation reminder

All results in this notebook are descriptive findings from the sample dataset. The delay thresholds are project-defined categories: on time (≤15 minutes), minor (16–60), major (61–180), and severe (>180). Records with missing arrival delay are labelled `Unknown` rather than being treated as on-time.